# 실제 Agent HTTP 통합

OpenAI 모델과 조회 도구가 있는 LangChain `create_agent`를 FastAPI에서 실제 호출합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 키 존재 확인

상위 폴더의 `.env`를 읽고 키 값은 출력하지 않습니다.

In [2]:
import os
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(usecwd=True))
has_key = bool(os.getenv('OPENAI_API_KEY'))
print('OPENAI_API_KEY 설정:', has_key)
assert has_key

OPENAI_API_KEY 설정: True


## 실제 ainvoke HTTP 요청

Agent는 운영 시간을 도구로 확인한 뒤 한 문장으로 답합니다.

In [3]:
from fastapi.testclient import TestClient
from agent_service.app import app
client = TestClient(app)
headers = {'Authorization': 'Bearer local-demo-token'}
body = {'question': '한빛도서관 월요일 운영 시간은?', 'thread_id': 'live-1'}
response = client.post('/v1/chat/live', headers=headers, json=body)
print(response.status_code, response.json()['answer'])
assert response.status_code == 200
assert '18:00' in response.json()['answer']

200 한빛도서관 월요일 운영 시간은 09:00~18:00입니다.


## 도구 호출 궤적 확인

같은 실제 Agent의 메시지에서 도구 이름, 입력, 도구 출력을 확인합니다.

In [4]:
from langchain_core.messages import ToolMessage
from agent_service.live_agent import build_live_agent
live_agent = build_live_agent()
trace = await live_agent.ainvoke({'messages': [
    {'role': 'user', 'content': body['question']}
]})
tool_call = next(call for message in trace['messages']
    for call in getattr(message, 'tool_calls', [])
    if call['name'] == 'lookup_library_hours')
tool_result = next(message for message in trace['messages']
    if isinstance(message, ToolMessage))
print(tool_call['name'], tool_call['args'], tool_result.content)
assert tool_call['args']['day'] == '월요일'
assert tool_result.content == '09:00~18:00'

lookup_library_hours {'day': '월요일'} 09:00~18:00


## 실제 astream HTTP 요청

같은 Agent의 모델 메시지 청크를 SSE token 이벤트로 받습니다.

In [5]:
import json, re
stream = client.post('/v1/chat/live/stream', headers=headers, json=body)
blocks = [b for b in re.split(r'\r?\n\r?\n', stream.text) if b]
tokens = []
for block in blocks:
    if block.startswith('event: token'):
        data_line = next(line for line in block.splitlines() if line.startswith('data: '))
        tokens.append(json.loads(data_line[6:])['token'])
live_answer = ''.join(tokens)
print(stream.status_code, live_answer)
assert stream.status_code == 200
assert blocks[-1].startswith('event: done')
assert '18:00' in live_answer

200 한빛도서관 월요일 운영 시간은 09:00~18:00입니다.


## 실험 결과 해석

결정적 그래프와 실제 모델 Agent의 출력 성질을 구분합니다.

In [6]:
token_events = stream.text.count('event: token')
print('실제 모델 token 이벤트:', token_events)
print('결정적 그래프가 아닌 create_agent 호출: True')
assert token_events > 0

실제 모델 token 이벤트: 20
결정적 그래프가 아닌 create_agent 호출: True
